In [1]:
import numpy as np
from typing import Optional, Union, Tuple, List, Dict

In [19]:
# haplotype functions

def _h2_single(counts: np.ndarray, n_valid: Optional[np.ndarray]=None) -> np.ndarray:
    """Compute H₂ for a single population"""
    c1, c2, c3, c4 = counts[:, 0], counts[:, 1], counts[:, 2], counts[:, 3]
    n = np.sum(counts, axis=1)

    numer = c1 * c4 + c2 * c3
    denom = n * (n - 1) / 2
    
    valid_mask = n >= 2
    result = np.zeros_like(n, dtype=np.float64)
    result[valid_mask] = numer[valid_mask] / denom[valid_mask]
    return result


def _h2_between(counts: np.ndarray,
                pop1_idx: int,
                pop2_idx: int,
                n_valid: Optional[np.ndarray]=None) -> np.ndarray:
    """Compute H₂ between two populations"""
    # Get indices to extract counts for each population
    start1 = pop1_idx * 4
    start2 = pop2_idx * 4

    c11, c12, c13, c14 = counts[:, start1], counts[:, start1+1], counts[:, start1+2], counts[:, start1+3]
    c21, c22, c23, c24 = counts[:, start2], counts[:, start2+1], counts[:, start2+2], counts[:, start2+3]

    if n_valid is not None:
        if isinstance(n_valid, tuple):
            n1 = n_valid[0] if n_valid[0] is not None else np.sum(counts[:, start1:start1+4], axis=1)
            n2 = n_valid[1] if n_valid[1] is not None else np.sum(counts[:, start2:start2+4], axis=1)
        elif hasattr(n_valid, 'ndim') and n_valid.ndim == 2:
            n1 = n_valid[:, pop1_idx]
            n2 = n_valid[:, pop2_idx]
        else:
            # Assume n_valid is for between-population pairs
            n1 = n_valid
            n2 = n_valid
    else:
        n1 = np.sum(counts[:, start1:start1 + 4], axis=1)
        n2 = np.sum(counts[:, start2:start2 + 4], axis=1)

    numer = c11 * c24 + c21 * c14 + c12 * c23 + c22 * c13
    denom = n1 * n2
    
    valid_mask = (n1 >= 2) & (n2 >= 2)
    result = np.zeros_like(n1, dtype=np.float64)
    result[valid_mask] = numer[valid_mask] / denom[valid_mask]
    return result

In [33]:
_PopDataGeno(np.array([[1, 1, 1, 1, 1, 1, 1, 1, 1]]), None).n

array([9.])

In [48]:
# genotype functions


class _PopDataGeno:
    """Precomputed per-population arrays for genotype-based LD computation.

    Stores 9 genotype configuration counts and derived frequency quantities.
    """
    __slots__ = ('g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9', 'n')

    def __init__(self, counts, n_valid):
        self.g1 = counts[:, 8].astype(np.float64)  # n22
        self.g2 = counts[:, 7].astype(np.float64)  # n21
        self.g3 = counts[:, 6].astype(np.float64)  # n20
        self.g4 = counts[:, 5].astype(np.float64)  # n12
        self.g5 = counts[:, 4].astype(np.float64)  # n11
        self.g6 = counts[:, 3].astype(np.float64)  # n10
        self.g7 = counts[:, 2].astype(np.float64)  # n02
        self.g8 = counts[:, 1].astype(np.float64)  # n01
        self.g9 = counts[:, 0].astype(np.float64)  # n00
        if n_valid is not None:
            self.n = n_valid.astype(np.float64)
        else:
            self.n = (self.g1 + self.g2 + self.g3 + self.g4 + self.g5
                      + self.g6 + self.g7 + self.g8 + self.g9)


def _safe_div(numer, denom, valid):
    """Element-wise division returning 0 where *valid* is False."""
    return np.where(valid, numer / np.maximum(denom, 1), 0.0)


def h2_geno_single(p):
    """Compute H₂ for a single population from _PopDataGeno"""
    g1, g2, g3, g4, g5, g6, g7, g8, g9 = (
        p.g1, p.g2, p.g3, p.g4, p.g5, p.g6, p.g7, p.g8, p.g9)
    n = p.n
    numer = (
        g1 * g5
        + 2 * g1 * g6
        + 2 * g1 * g8
        + 4 * g1 * g9
        + g2 * g4
        + g2 * g5
        + g2 * g6
        + 2 * g2 * g7
        + 2 * g2 * g8
        + 2 * g2 * g9
        + 2 * g3 * g4
        + g3 * g5
        + 4 * g3 * g7
        + 2 * g3 * g8
        + g4 * g5
        + 2 * g4 * g6
        + g4 * g8
        + 2 * g4 * g9
        + g5 * (g5 + 1) / 2
        + g5 * g6
        + g5 * g7
        + g5 * g8
        + g5 * g9
        + 2 * g6 * g7
        + g6 * g8
        )
    denom = n * (2 * n - 1)
    valid = n >= 2
    return _safe_div(numer, denom, valid)


def h2_geno_between(p1, p2):
    """Compute H₂ between two populations from two _PopDataGeno instances"""
    g11, g12, g13, g14, g15, g16, g17, g18, g19 = (
        p1.g1, p1.g2, p1.g3, p1.g4, p1.g5, p1.g6, p1.g7, p1.g8, p1.g9)
    g21, g22, g23, g24, g25, g26, g27, g28, g29 = (
        p2.g1, p2.g2, p2.g3, p2.g4, p2.g5, p2.g6, p2.g7, p2.g8, p2.g9)
    n1 = p1.n
    n2 = p2.n
    numer = (
        g11 * g25 / 4
        + g11 * g26 / 2
        + g11 * g28 / 2
        + g11 * g29
        + g12 * g24 / 4
        + g12 * g25 / 4
        + g12 * g26 / 4
        + g12 * g27 / 2
        + g12 * g28 / 2
        + g12 * g29 / 2
        + g13 * g24 / 2
        + g13 * g25 / 4
        + g13 * g27
        + g13 * g28 / 2
        + g14 * g22 / 4
        + g14 * g23 / 2
        + g14 * g25 / 4
        + g14 * g26 / 2
        + g14 * g28 / 4
        + g14 * g29 / 2
        + g15 * g21 / 4
        + g15 * g22 / 4
        + g15 * g23 / 4
        + g15 * g24 / 4
        + g15 * g25 / 4
        + g15 * g26 / 4
        + g15 * g27 / 4
        + g15 * g28 / 4
        + g15 * g29 / 4
        + g16 * g21 / 2
        + g16 * g22 / 4
        + g16 * g24 / 2
        + g16 * g25 / 4
        + g16 * g27 / 2
        + g16 * g28 / 4
        + g17 * g22 / 2
        + g17 * g23
        + g17 * g25 / 4
        + g17 * g26 / 2
        + g18 * g21 / 2
        + g18 * g22 / 2
        + g18 * g23 / 2
        + g18 * g24 / 4
        + g18 * g25 / 4
        + g18 * g26 / 4
        + g19 * g21
        + g19 * g22 / 2
        + g19 * g24 / 2
        + g19 * g25 / 4
        )
    denom = n1 * n2    
    valid = (n1 >= 1) & (n2 >= 1)
    return _safe_div(numer, denom, valid)


def h2_geno_between_alt(p1, p2):
    """Compute H₂ between two populations from two _PopDataGeno instances"""
    g11, g12, g13, g14, g15, g16, g17, g18, g19 = (
        p1.g1, p1.g2, p1.g3, p1.g4, p1.g5, p1.g6, p1.g7, p1.g8, p1.g9)
    g21, g22, g23, g24, g25, g26, g27, g28, g29 = (
        p2.g1, p2.g2, p2.g3, p2.g4, p2.g5, p2.g6, p2.g7, p2.g8, p2.g9)
    n1 = p1.n
    n2 = p2.n
    c11 = g11 + g12 / 2 + g14 / 2 + g15 / 4
    c12 = g12 / 2 + g13 + g15 / 4 + g16 / 2
    c13 = g14 / 2 + g15 / 4 + g17 + g18 / 2
    c14 = g15 / 4 + g16 / 2 + g18 / 2 + g19
    c21 = g21 + g22 / 2 + g24 / 2 + g25 / 4
    c22 = g22 / 2 + g23 + g25 / 4 + g26 / 2
    c23 = g24 / 2 + g25 / 4 + g27 + g28 / 2
    c24 = g25 / 4 + g26 / 2 + g28 / 2 + g29
    numer = c11 * c24 + c21 * c14 + c12 * c23 + c22 * c13
    denom = n1 * n2
    valid = (n1 >= 1) & (n2 >= 1)
    return _safe_div(numer, denom, valid)

In [47]:
p1 = _PopDataGeno(np.array([[2, 0, 0, 0, 2, 0, 0, 0, 0]]), None)
p2 = _PopDataGeno(np.array([[0, 0, 0, 0, 2, 0, 0, 0, 2]]), None)
print(h2_geno_between(p1, p2), h2_geno_between_alt(p1, p2))
%timeit h2_geno_between(p1, p2)
%timeit h2_geno_between_alt(p1, p2)

[0.4375] [0.4375]
58.1 μs ± 294 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
27.4 μs ± 103 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [50]:
h2_geno_single(p2)

array([0.25])